# Anomaly detection in Snowflake, plotted with `nixtla_client`

Detection runs in Snowflake through `NIXTLA_DETECT_ANOMALIES`; the charts are
drawn locally by `nixtla_client.plot`, which runs entirely in this notebook and
makes no API call.

The procedure returns the same information as `NixtlaClient.detect_anomalies`
under different column names, so `anomaly.py` renames them back. Add that file
to this notebook (**Files** in the sidebar) before running the next cell.

## Imports

In [ ]:
try:
    from nixtla import NixtlaClient
except ImportError:
    !pip install -q nixtla
    from nixtla import NixtlaClient

from anomaly import detect_anomalies, load_actuals, to_anomalies_df

## Instantiate NixtlaClient

In [ ]:
from snowflake.snowpark.secrets import get_generic_secret_string

nixtla_client = NixtlaClient(
    api_key=get_generic_secret_string("demo/public/nixtla_api_key")
)

## The data

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

INPUT_TABLE = "DEMO.PUBLIC.EXAMPLE_ANOMALY_DATA"
FREQ = "D"
MODEL = "timegpt-1"

In [ ]:
actuals = load_actuals(session, INPUT_TABLE)
nixtla_client.plot(actuals)

## Detect anomalies

`detect_anomalies` calls the procedure and hands back a frame shaped like the
client's own output. `plot` reads the level, the flags and the model name off
that frame, so nothing else needs passing.

The procedure is looked up as `NIXTLA_DETECT_ANOMALIES` in this notebook's
current database and schema. If you installed it elsewhere, add
`procedure="<db>.<schema>.NIXTLA_DETECT_ANOMALIES"`.

In [ ]:
anomalies = detect_anomalies(session, INPUT_TABLE, level=99, procedure="DEMO.PUBLIC.NIXTLA_DETECT_ANOMALIES", freq=FREQ, model=MODEL)
anomalies.head()

In [ ]:
nixtla_client.plot(actuals, anomalies)

### A different confidence level

A wider interval flags fewer points.

In [ ]:
wider = detect_anomalies(session, INPUT_TABLE, level=99.9, procedure="DEMO.PUBLIC.NIXTLA_DETECT_ANOMALIES", freq=FREQ, model=MODEL)

for name, df in [("99", anomalies), ("99.9", wider)]:
    print(f"level={name:<5} {int(df['anomaly'].sum())} anomalies of {len(df)} points")

nixtla_client.plot(actuals, wider)

## Plotting a result from a SQL cell

A SQL cell exposes its result to Python under the pointer name shown on the
cell -- `dataframe_1` for the first one, double-click to rename. `to_anomalies_df`
renames the columns the same way `detect_anomalies` does; pass the level the SQL
ran at, since the procedure strips it from the interval column names.

In [ ]:
CALL DEMO.PUBLIC.NIXTLA_DETECT_ANOMALIES(
    INPUT_DATA => 'DEMO.PUBLIC.EXAMPLE_ANOMALY_DATA',
    PARAMS => OBJECT_CONSTRUCT(
        'level', 99.9,
        'freq',  'D',
        'model', 'timegpt-1'
    )
);

In [ ]:
nixtla_client.plot(actuals, to_anomalies_df(dataframe_1, level=99.9))